In [26]:
import re
import string
import pandas as pd
import torch 
import datasets as ds
import os
import random
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# Define your target device
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

# get the data
d = ds.load_dataset("sentence-transformers/natural-questions", split="train", streaming=True)
data = d.shuffle(seed=random.randint(0,20000000), buffer_size=10000).take(2000).to_pandas()
data = data.drop('answer',axis=1) # drops the answers
data['query'] = data['query'].str.replace(r"[^a-zA-Z0-9'/-_ ]", '', regex=True) # remove non alphanum chars
data['query'] = data['query'].sample(frac=1).reset_index(drop=True)
print(data)

PAD = '\\' # padding  (backlash literal)

word_unique = set(data['query'].str.lower().str.cat(sep=' ').split())
#print(word_unique)
word_unique.add('<BOS>') # beginning of word 

word_to_id = {word: index for index, word in enumerate(word_unique)}
id_to_word = {value: key for key, value in word_to_id.items()} 

char_vocab = f"abcdefghijklmnopqrstuvwxyz0123456789'/-_{PAD}"
char_to_id = {c: i for i, c in enumerate(char_vocab)}
id_to_char = {value: key for key, value in char_to_id.items()} 
CHARS_UNIQUE = len(char_vocab)  # alpha lowercase + num + <PAD> (current word is empty)


                                                  query
0        who plays the grandmother in switched at birth
1         when does ellie find out chuck is still a spy
2      where does the name accrington stanley come from
3         when was the first cadbury chocolate bar made
4                when does the 5sos album come out 2018
...                                                 ...
1995   actress who plays dr obrecht on general hospital
1996            who did the tennessee titans used to be
1997         who is alex karev dating on grey's anatomy
1998         who sang would you like to swing on a star
1999  what are the responsibilities of the transport...

[2000 rows x 1 columns]


In [27]:
from torch.utils.data import Dataset
class HybridTextDataset(Dataset):
    def __init__(self, data:pd.Series, word_to_id:dict, char_to_id:dict):
        # We now expect a single dictionary of parallel arrays
        
        # self.words = data_dict["words"]
        # self.chars = data_dict["chars"]
        # self.targets = data_dict["targets"]
        self.words = []
        self.chars = []
        self.targets = []
        
        for sentence in data.tolist(): # every sentence in dset
            words_list = sentence.split() # get the words , gets rid of non alphanum
            for word_idx, word in enumerate(words_list): # every word in sentence
                
                # insert the PAD 
                
                # append previous words, bos if none
                self.words.append([word_to_id[w] for w in words_list[:word_idx]] if word_idx > 0 else [word_to_id['<BOS>']])   
                self.chars.append([char_to_id[PAD]]) # appends the pad 
                self.targets.append(word_to_id[word]) # appends the word 

                    
                chrs = [char_to_id[c] for c in word[:-1]] # gets ids of the characters except the last one
                
                for char_idx in range(len(chrs)):
                    # append previous words, bos if none
                    self.words.append([word_to_id[w] for w in words_list[:word_idx]] if word_idx > 0 else [word_to_id['<BOS>']])   
                    
                    self.chars.append(chrs[:char_idx+1]) # add all characters until this point 
                    self.targets.append(word_to_id[word]) # appends the word 
        # for i in range(100):
        #     print([id_to_word[w] for w in self.words[i]], 
        #           [id_to_char[w] for w in self.chars[i]],
        #           id_to_word[self.targets[i]],)
                
        # print(self.words)
        # print(self.chars)
        # print(self.targets)
            # append list of words before the currently typed word
            # if no words, insert the beginning of string flag
            #self.words.append([word_unique[word] for word in words_list[:-1]] if len(words_list)>1 else [word_unique['<BOS>']]) # append previous words
            
            
            
        

    def __len__(self):
        # All arrays should be the exact same length
        return len(self.targets)

    def __getitem__(self, idx):
        # We index into the parallel arrays directly
        word_tensor = torch.tensor(self.words[idx], dtype=torch.long)
        char_tensor = torch.tensor(self.chars[idx], dtype=torch.long)
        target_tensor = torch.tensor(self.targets[idx], dtype=torch.long)
        
        return word_tensor, char_tensor, target_tensor

In [28]:
from torch import nn
class HybridPredictiveModel(nn.Module):
    '''
    whats going on?
    dual-input NN : takes the characters (or lack thereof) of a currently typed word + previous words to estimate the next word
    
    characters:
    GRU recurrent NN, reads characters one by one , returning mathematical summary of the word
    
    words:
    embedding layer finds matching words from a dictionary, value being a dense vector
    
    words + characters get concatenated and passed to LSTM (long short-term memory)
    LSTM remembers long-term dependencies (long queries)
    
    
    
    
    '''
    
    def __init__(self, n_word_vocab, n_char_vocab):
        super(HybridPredictiveModel, self).__init__()
        self.embedding_dim = 128
        self.lstm_size = 256
        self.num_layers = 3
        self.unique_words = n_word_vocab
        self.unique_chars = n_char_vocab
        # 1. Main Word Pathway
        self.word_embedding = nn.Embedding(n_word_vocab, self.embedding_dim)

        # 2. Character Pathway (The "Partial Word" Encoder)
        self.char_embedding = nn.Embedding(n_char_vocab, 32)
        # A lightweight GRU just to squash characters into a single vector
        self.char_encoder = nn.GRU(
            input_size=32, 
            hidden_size=self.embedding_dim, # Must match word embedding dim!
            batch_first=True
        )

        # 3. Main Sequence Modeler
        self.lstm = nn.LSTM(
            input_size=self.embedding_dim,
            hidden_size=self.lstm_size,
            num_layers=self.num_layers,
            dropout=0.2,
            batch_first=True # Recommended to make tensor dimensions easier (Batch, Seq, Feature)
        )
        
        self.fc = nn.Linear(self.lstm_size, n_word_vocab)

    def forward(self, completed_words_x, current_chars_x, prev_state):
        # completed_words_x shape: (batch_size, word_sequence_length)
        # current_chars_x shape: (batch_size, char_sequence_length)

        # -- PROCESS WORDS --
        # Shape becomes: (batch_size, word_seq_len, 128)
        word_vectors = self.word_embedding(completed_words_x)

        # -- PROCESS CHARACTERS --
        char_embeds = self.char_embedding(current_chars_x)
        # Pass characters through the GRU. We only care about the final hidden state
        _, final_char_state = self.char_encoder(char_embeds)
        
        # The hidden state comes out as (num_layers, batch, hidden_size). 
        # We reshape it to act like a single "word" in the sequence: (batch, 1, 128)
        char_vector = final_char_state[-1].unsqueeze(1)

        # -- THE MERGE --
        # Append the character vector to the end of the word sequence
        # Shape becomes: (batch_size, word_seq_len + 1, 128)
        combined_sequence = torch.cat([word_vectors, char_vector], dim=1)

        # -- MAIN LSTM --
        output, state = self.lstm(combined_sequence, prev_state)
        
        # We usually only care about predicting the very next token, 
        # so we grab the output from the final time step
        final_timestep_output = output[:, -1, :] 
        
        logits = self.fc(final_timestep_output)

        return logits, state

    def init_state(self, sequence_length):
        return (
            torch.zeros(self.num_layers, \
                        sequence_length, self.lstm_size).to(device),
            torch.zeros(self.num_layers, \
                        sequence_length, self.lstm_size).to(device)
        )

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, Dataset
from torch.nn.utils.rnn import pad_sequence
    
def hybrid_collate_fn(batch):
    # 'batch' is a list of tuples returned by __getitem__
    words_list = [item[0] for item in batch]
    chars_list = [item[1] for item in batch]
    targets_list = [item[2] for item in batch]
    
    # Pad the sequences
    batched_words = pad_sequence(words_list, batch_first=True, padding_value=0)
    batched_chars = pad_sequence(chars_list, batch_first=True, padding_value=0)
    
    # Stack the targets (single integer predictions)
    batched_targets = torch.stack(targets_list)
    
    return batched_words, batched_chars, batched_targets

# Hyperparameters
sequence_length = 12
batch_size = 256
learning_rate = 0.022
num_epochs = 40

# Create the dataset
dataset = HybridTextDataset(data['query'], word_to_id, char_to_id)

# Split the dataset into training and validation sets
total_len = len(dataset)
split_idx = int(0.8 * total_len)
train_indices = list(range(0, split_idx))
val_indices = list(range(split_idx, total_len))

# Create Subsets
train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

# Create data loaders
g = torch.Generator(device=device)
train_loader = DataLoader(train_dataset, 
                          batch_size=batch_size, 
                          collate_fn=hybrid_collate_fn,
                          generator=g,
                          shuffle=True)

val_loader = DataLoader(val_dataset, 
                        batch_size=batch_size,
                        generator=g,
                        collate_fn=hybrid_collate_fn,
                        shuffle=True)

# Create the model (Removed the duplicate initialization)
# Assuming CHARS_UNIQUE and len(char_vocab) represent the same thing
model = HybridPredictiveModel(len(word_unique), CHARS_UNIQUE).to(device) 

# Print Model Information
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("-" * 50)
print(f"Model Architecture Initialized")
print(f"Total Trainable Parameters: {total_params:,}")
print(f"Training on device: {device}")
print("-" * 50)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

# Training loop
for epoch in range(num_epochs):
    epoch_start_time = time.time()
    
    # --- TRAINING PHASE ---
    model.train()
    total_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batched_words, batched_chars, targets in train_loader:
        # Move data to GPU
        batched_words = batched_words.to(device)
        batched_chars = batched_chars.to(device)
        targets = targets.to(device)
        
        current_batch_size = batched_words.size(0)
        
        # Initialize fresh hidden states
        hidden_state = torch.zeros(model.num_layers, current_batch_size, model.lstm_size).to(device)
        cell_state = torch.zeros(model.num_layers, current_batch_size, model.lstm_size).to(device)
        prev_state = (hidden_state, cell_state)
        
        # Forward Pass
        optimizer.zero_grad()
        logits, _ = model(batched_words, batched_chars, prev_state)
        
        # Calculate Loss & Backpropagate
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        # Update metrics (FIXED: Added loss tracking)
        total_loss += loss.item()
        
        # Calculate training accuracy
        _, predicted = torch.max(logits.data, 1)
        total_train += targets.size(0)
        correct_train += (predicted == targets).sum().item()

    average_train_loss = total_loss / len(train_loader)
    train_accuracy = 100 * correct_train / total_train

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for batched_words, batched_chars, targets in val_loader:
            batched_words = batched_words.to(device)
            batched_chars = batched_chars.to(device)
            targets = targets.to(device)
            
            current_batch_size = batched_words.size(0)
            
            hidden_state = torch.zeros(model.num_layers, current_batch_size, model.lstm_size).to(device)
            cell_state = torch.zeros(model.num_layers, current_batch_size, model.lstm_size).to(device)
            val_prev_state = (hidden_state, cell_state)

            outputs, _ = model(batched_words, batched_chars, val_prev_state)

            # Calculate validation loss and accuracy
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total_val += targets.size(0)
            correct_val += (predicted == targets).sum().item()

    average_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct_val / total_val
    
    epoch_duration = time.time() - epoch_start_time

    # --- PRINT EPOCH SUMMARY ---
    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Time: {epoch_duration:.2f}s")
    print(f"  Train -> Loss: {average_train_loss:.4f} | Accuracy: {train_accuracy:.2f}%")
    print(f"  Val   -> Loss: {average_val_loss:.4f} | Accuracy: {val_accuracy:.2f}%")
    print("-" * 50)

--------------------------------------------------
Model Architecture Initialized
Total Trainable Parameters: 3,061,851
Training on device: cuda
--------------------------------------------------
Epoch [01/40] | Time: 8.37s
  Train -> Loss: 6.5066 | Accuracy: 9.90%
  Val   -> Loss: 6.7477 | Accuracy: 11.79%
--------------------------------------------------
Epoch [02/40] | Time: 8.31s
  Train -> Loss: 5.1801 | Accuracy: 14.45%
  Val   -> Loss: 6.6044 | Accuracy: 15.46%
--------------------------------------------------
Epoch [03/40] | Time: 8.62s
  Train -> Loss: 4.3481 | Accuracy: 17.77%
  Val   -> Loss: 6.7381 | Accuracy: 16.74%
--------------------------------------------------
Epoch [04/40] | Time: 8.46s
  Train -> Loss: 3.9852 | Accuracy: 20.22%
  Val   -> Loss: 6.8099 | Accuracy: 16.69%
--------------------------------------------------
Epoch [05/40] | Time: 8.54s
  Train -> Loss: 3.7448 | Accuracy: 22.09%
  Val   -> Loss: 6.8897 | Accuracy: 16.86%
-------------------------------

In [ ]:
# Input a sentence
input_sentence = "why are the volcanoes"

# Preprocess the input sentence
input_indexes = [word_to_id[word] for word in input_sentence.split() if word in word_to_id]
input_tensor = torch.tensor(input_indexes, dtype=torch.long).unsqueeze(0).to(device)

# Generate the next word
model.eval()
with torch.no_grad():
    current_batch_size = input_tensor.size(0)
    hidden_state = torch.zeros(model.num_layers, current_batch_size, model.lstm_size).to(device)
    cell_state = torch.zeros(model.num_layers, current_batch_size, model.lstm_size).to(device)
    prev_state = (hidden_state, cell_state)
    
    # Create dummy character tensor
    char_tensor = torch.tensor([[char_to_id[PAD]]], dtype=torch.long).to(device)
    
    outputs, _ = model(input_tensor, char_tensor, prev_state)

    predicted_index = torch.argmax(outputs[0]).item()    print("Predicted Next Word:", predicted_word)

    predicted_word = id_to_word[predicted_index]    print("Input Sentence:", input_sentence)

    # Print the predicted word

SyntaxError: invalid syntax (1068366375.py, line 21)